# Language Models for Text Classification with the AG News Dataset — Student Version

In this notebook, you will progressively build a small prompting-based text classification pipeline with the **AG News** dataset.

## Learning goals
- load and inspect a text classification dataset
- design **zero-shot** and **few-shot** prompts
- query **two LLM APIs** (**Groq** and **Mistral**)
- normalize model outputs
- evaluate predictions on a small test subset
- reflect on strengths and limitations of prompting for classification


## 1. Setup

This notebook should use:
- `datasets` for AG News
- `pandas` and `matplotlib` for quick inspection
- `groq` for one API
- `mistralai` for a second API

> Keep API keys out of the notebook when possible.

In [ ]:
# TODO: uncomment and run if needed
# !pip install -q datasets pandas matplotlib groq mistralai

In [ ]:
import os
import re
import json
import random
import time
from typing import List, Dict, Optional

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from openai import OpenAI
from groq import Groq

In [ ]:
SEED = 42

# TODO: set the random seed

# Small subsets keep the demo fast and avoid exhausting free quotas.
TRAIN_SUBSET = 24
TEST_SUBSET = 20

LABELS = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}

# TODO:
# 1) set your API keys in the environment before running
# 2) read them with os.getenv(...)
# 3) print a quick check (True / False)

# Example in a terminal:
# export GROQ_API_KEY="..."
# export MISTRAL_API_KEY="..."

GROQ_API_KEY = ...
MISTRAL_API_KEY = ...

GROQ_MODEL = ...
MISTRAL_MODEL = ...

print("Groq key loaded:", ...)
print("Mistral key loaded:", ...)

## 2. Load and inspect AG News

AG News contains four topic classes:
1. World
2. Sports
3. Business
4. Sci/Tech

Each example contains a piece of text and a numeric label.

In [ ]:
print("Loading AG News...")
dataset = load_dataset("ag_news")

train_data = dataset["train"].shuffle(seed=SEED)
test_data = dataset["test"].shuffle(seed=SEED)

if TRAIN_SUBSET is not None:
    train_data = train_data.select(range(TRAIN_SUBSET))
if TEST_SUBSET is not None:
    test_data = test_data.select(range(TEST_SUBSET))

print(dataset)
print("Train subset size:", len(train_data))
print("Test subset size:", len(test_data))

In [ ]:
# TODO: convert the subsets to pandas DataFrames
df_train = ...
df_test = ...

# TODO: map numeric labels to label names
df_train["label_name"] = ...
df_test["label_name"] = ...

# TODO: inspect a few rows
display(...)
display(...)

In [ ]:
# TODO: plot the class distribution in the train subset

# Your code here

## 3. Build prompts

We want the model to return **only one** of these labels:
- `World`
- `Sports`
- `Business`
- `Sci/Tech`

### TODOs
- understand why constrained output matters
- compare **zero-shot** and **few-shot**
- decide what makes a good support example

In [ ]:
def format_example(example):
    # TODO:
    # return a formatted string version of one AG News example
    # You may include:
    # - the title
    # - the full text
    # Keep the format simple and consistent.
    return ...

In [ ]:
def build_zero_shot_prompt(example):
    # TODO:
    # create a prompt that:
    # 1) tells the model it is a news classifier
    # 2) lists the four possible labels
    # 3) asks for exactly one label
    # 4) includes the news example
    # 5) avoids explanations in the answer
    prompt = f"""
    TODO
    """
    return prompt.strip()

In [ ]:
def build_few_shot_prompt(example, support_examples):
    # TODO:
    # create a few-shot prompt using support examples
    #
    # Requirements:
    # - reuse the same instruction style as zero-shot
    # - include labeled examples before the test item
    # - keep formatting identical across examples
    # - finish with the unlabeled target example
    #
    # Optional challenge:
    # - try 1 example per class
    # - later try 2 examples per class

    support_block = ""

    # TODO: loop through support_examples and build the support block

    prompt = f"""
    TODO
    {support_block}

    TODO: add the target example here
    """
    return prompt.strip()

## 4. API helpers

We will compare:
1. **Groq**
2. **Mistral**

Both functions should receive a prompt and return the model's raw text output.

In [ ]:
# TODO: initialize the Groq client
# groq_client = ...

def ask_groq(prompt, model=GROQ_MODEL):
    # TODO:
    # complete the API call
    # Hints:
    # - use a chat completion
    # - pass the prompt as a user message
    # - use temperature=0 for deterministic behavior
    # - keep max_tokens small
    response = ...

    # TODO: return only the text content
    return ...

In [ ]:
# TODO: initialize the Mistral client
# mistral_client = ...

def ask_mistral(prompt, model=MISTRAL_MODEL):
    # TODO:
    # complete the API call using the official Mistral client
    # Hints:
    # - use client.chat.complete(...)
    # - pass one user message
    # - keep the answer short
    response = ...

    # TODO: return only the text content
    return ...

In [ ]:
VALID_LABELS = set(LABELS.values())

def normalize_prediction(text):
    # TODO:
    # convert a raw model answer into one of the valid labels or None
    #
    # Suggested steps:
    # 1) handle None
    # 2) strip whitespace
    # 3) check exact match
    # 4) fallback: search for one valid label inside a longer sentence
    return ...

In [ ]:
def evaluate_predictions(predictions, gold_ids):
    # TODO:
    # return a DataFrame with:
    # - gold_id
    # - gold_label
    # - prediction
    # - correct (True/False)

    rows = []

    # TODO: fill rows

    return ...

## 5. Zero-shot classification

In zero-shot prompting, the model gets **instructions only**, with **no labeled examples**.

### TODOs
- test each API on a very small subset first
- inspect the raw outputs
- check whether the model respects the required format

In [ ]:
def run_experiment(
    dataset_split,
    ask_fn,
    prompt_builder,
    support_examples=None,
    sleep_seconds=0.0,
):
    predictions = []
    raw_outputs = []

    for example in dataset_split:
        # TODO:
        # - build the prompt
        # - call the API
        # - normalize the answer
        # - store both raw and normalized outputs
        # - handle errors with try/except
        raw = ...
        pred = ...

        raw_outputs.append(raw)
        predictions.append(pred)

        # TODO: optional pause to respect rate limits
        # time.sleep(...)

    return predictions, raw_outputs

In [ ]:
# TODO:
# 1) create a tiny test slice (for example, 5 items)
# 2) extract the gold labels
# 3) run zero-shot with Groq
# 4) run zero-shot with Mistral
# 5) display evaluation tables
# 6) print the raw outputs to inspect formatting

tiny_test = ...
gold_tiny = ...

zs_groq_preds, zs_groq_raw = ...
zs_mistral_preds, zs_mistral_raw = ...

display(...)
display(...)

print("Groq raw outputs:", ...)
print("Mistral raw outputs:", ...)

## 6. Few-shot classification

In few-shot prompting, we include a small number of labeled examples in the prompt.

### TODOs
- build a support set
- start with **one example per class**
- later try more examples
- reflect on whether support example choice changes the results

In [ ]:
# TODO:
# create a support set from the training subset
#
# Suggested strategy:
# - pick one example for each label
# - store them in support_examples
# - verify that all 4 labels are covered

support_examples = []
seen = set()

# Your code here

for ex in support_examples:
    # TODO: print the label name and formatted example
    print(...)
    print(...)
    print("-" * 80)

In [ ]:
# TODO:
# run few-shot classification on the full test subset with both APIs

gold_test = ...

fs_groq_preds, fs_groq_raw = ...
fs_mistral_preds, fs_mistral_raw = ...

df_fs_groq = ...
df_fs_mistral = ...

display(...)
display(...)

## 7. Compare zero-shot vs few-shot

You should now have four settings:
- Groq zero-shot
- Groq few-shot
- Mistral zero-shot
- Mistral few-shot

### TODOs
- compute accuracy for each setting
- compare the numbers
- identify at least 2 error cases

In [ ]:
def accuracy(preds, gold_ids):
    # TODO:
    # compute simple accuracy
    return ...

results = {
    # TODO: fill this dictionary with the four accuracies
    # "Groq zero-shot": ...,
    # "Groq few-shot": ...,
    # "Mistral zero-shot": ...,
    # "Mistral few-shot": ...,
}

results_df = ...
display(results_df)

# TODO: optional bar plot of the four accuracies

## 8. Reflection questions

Answer briefly in your own words.

1. Did few-shot improve performance?
2. Which classes seemed easiest or hardest?
3. Were the outputs always clean and easy to parse?
4. How sensitive were the results to prompt wording?
5. What are the limitations of API-based prompting for classification?
6. When would you prefer a supervised classifier instead?

## 9. Optional extensions

Choose one or more:

- Try **2 support examples per class**
- Compare **headline only** vs **full text**
- Force the model to output a **JSON object**
- Build a **confusion matrix**
- Measure **latency** per API
- Estimate **cost / quota usage**
- Intentionally use **biased support examples** and observe what changes

## 10. Final takeaway

Large language models can perform classification through prompting alone.

### Before finishing, make sure you completed:
- [ ] dataset loading
- [ ] zero-shot prompt
- [ ] few-shot prompt
- [ ] Groq helper
- [ ] Mistral helper
- [ ] normalization
- [ ] evaluation
- [ ] comparison
- [ ] reflection